# Final dataset: integration for Tableau

## Bringing Phases 2-6 together into the files the dashboard actually needs

This notebook doesn't produce new analysis. It integrates and validates
what Phases 2 through 6 already built, confirming the pieces actually fit
together (not assuming it, after two separate incidents earlier in this
project where an assumed-fine merge wasn't), and exports the final files
Tableau will read from.

---

# Dataset final: integración para Tableau

## Juntando las Fases 2-6 en los archivos que el dashboard realmente necesita

Este notebook no produce análisis nuevo. Integra y valida lo que las
Fases 2 a 6 ya construyeron, confirmando que las piezas realmente encajan
(no asumiéndolo, después de dos incidentes separados antes en este
proyecto donde una unión que se daba por buena no lo estaba), y exporta
los archivos finales desde los que va a leer Tableau.

In [1]:
import pandas as pd

population = pd.read_csv("../data/processed/population.csv")
role_metrics = pd.read_csv("../data/processed/role_metrics.csv")
role_scores = pd.read_csv("../data/processed/role_scores.csv")
role_specialists = pd.read_csv("../data/processed/role_specialists.csv")

for name, df in [("population", population), ("role_metrics", role_metrics),
                  ("role_scores", role_scores), ("role_specialists", role_specialists)]:
    print(f"{name}: {df.shape}")

# ¿Los ids de role_metrics y role_scores son exactamente el mismo conjunto?
ids_metrics = set(role_metrics["id"])
ids_scores = set(role_scores["id"])
print(f"\nEn role_metrics pero no en role_scores: {len(ids_metrics - ids_scores)}")
print(f"En role_scores pero no en role_metrics: {len(ids_scores - ids_metrics)}")

population: (5636, 29)
role_metrics: (2488, 45)
role_scores: (2488, 31)
role_specialists: (12, 6)

En role_metrics pero no en role_scores: 0
En role_scores pero no en role_metrics: 0


In [2]:
# role_scores tiene columnas que ya están en role_metrics (name, minutesPlayed,
# world_cups_played). Nos quedamos solo con las columnas nuevas de role_scores
# para no duplicar nada al unir.
overlap = set(role_metrics.columns) & set(role_scores.columns)
print(f"Columnas en común (aparte de 'id', que es la clave de unión): {overlap - {'id'}}")

new_cols_from_scores = ["id"] + [c for c in role_scores.columns if c not in role_metrics.columns]
print(f"\nColumnas que se agregan desde role_scores: {len(new_cols_from_scores) - 1}")

dataset_final = role_metrics.merge(role_scores[new_cols_from_scores], on="id", how="inner")
print(f"\nShape del dataset unido: {dataset_final.shape}")
print(f"Filas esperadas: 2488. Coincide: {len(dataset_final) == 2488}")

print(f"IDs duplicados: {dataset_final['id'].duplicated().sum()}")

Columnas en común (aparte de 'id', que es la clave de unión): {'minutesPlayed', 'name', 'world_cups_played'}

Columnas que se agregan desde role_scores: 27

Shape del dataset unido: (2488, 72)
Filas esperadas: 2488. Coincide: True
IDs duplicados: 0


In [3]:
# Confirmar que jugadores clave del proyecto llegaron completos, con datos
# de ambas fuentes, no solo de una.
key_players = ["Messi", "Maradona", "Cristiano Ronaldo", "Cruyff", "Zidane"]

for player in key_players:
    match = dataset_final[dataset_final["name"].str.contains(player, case=False, na=False)]
    if len(match) > 0:
        row = match.iloc[0]
        print(f"{row['name']}: minutos={row['minutesPlayed']}, "
              f"finisher_score={row['finisher_score']:.2f}, "
              f"n_elite_roles_90={row['n_elite_roles_90']}, "
              f"role_score_avg={row['role_score_avg']:.2f}")
    else:
        print(f"{player}: NO ENCONTRADO")

# Confirmar que no aparecieron columnas completamente vacías por un mal merge
empty_cols = dataset_final.columns[dataset_final.isna().all()].tolist()
print(f"\nColumnas completamente vacías (indicaría un merge roto): {empty_cols}")

Lionel Messi: minutos=3054.0, finisher_score=97.19, n_elite_roles_90=3, role_score_avg=92.28
Diego Armando Maradona: minutos=1940.0, finisher_score=88.46, n_elite_roles_90=2, role_score_avg=89.24
Cristiano Ronaldo: minutos=2206.0, finisher_score=96.64, n_elite_roles_90=1, role_score_avg=69.52
Johan Cruyff: minutos=630.0, finisher_score=89.45, n_elite_roles_90=2, role_score_avg=93.34
Zinedine Zidane: minutos=1109.0, finisher_score=86.87, n_elite_roles_90=1, role_score_avg=88.69

Columnas completamente vacías (indicaría un merge roto): []


## Exporting the final files

Two files leave this notebook: `dataset_final.csv` (2488 players, every
metric and score from Phases 3-5, one row per player) for the main
dashboard, and `role_specialists.csv` (already finalized in Phase 6,
copied over unchanged) for the radar chart. `population.csv` stays in
`data/processed/` as-is, available if Tableau needs to reference the full
5636-player population (e.g. to show how many players don't meet the
270-minute threshold), but isn't merged into the main file.

---

## Exportando los archivos finales

De este notebook salen dos archivos: `dataset_final.csv` (2488 jugadores,
cada métrica y puntaje de las Fases 3-5, una fila por jugador) para el
dashboard principal, y `role_specialists.csv` (ya finalizado en la Fase
6, copiado sin cambios) para el radar chart. `population.csv` se queda en
`data/processed/` tal cual, disponible si Tableau necesita referenciar la
población completa de 5636 jugadores (por ejemplo, para mostrar cuántos
no llegan al umbral de 270 minutos), pero no se fusiona al archivo
principal.

In [6]:
dataset_final.to_csv("../data/processed/dataset_final.csv", index=False)
print(f"dataset_final.csv guardado: {dataset_final.shape[0]} filas, {dataset_final.shape[1]} columnas")

print("\nArchivos finales en data/processed/:")
import os
for f in sorted(os.listdir("../data/processed")):
    print(f" - {f}")

dataset_final.csv guardado: 2488 filas, 72 columnas

Archivos finales en data/processed/:
 - .gitkeep
 - dataset_final.csv
 - population.csv
 - role_metrics.csv
 - role_scores.csv
 - role_specialists.csv


## Conclusion

Integrated Phases 2-6 into two files: `dataset_final.csv` (2488 players,
72 columns spanning career totals, per-90 rates, the four role scores,
elite-role counts at three thresholds, and the average role score) and
`role_specialists.csv` (12 labeled reference rows for the radar chart).
Verified before exporting, not assumed: `role_metrics` and `role_scores`
share the exact same 2488 player ids, the merge produced no duplicates
and no fully-empty columns, and five key players (Messi, Maradona,
Cristiano, Cruyff, Zidane) carry consistent values from both source files
matching every number already reported in Phases 3-6.

The full pipeline is now reproducible end to end from raw SofaScore data
to these two files: `01_extraction_coverage` confirms the raw data,
`02_population_build` cleans and aggregates it, `03_role_metrics`
normalizes and assigns it to roles, `04_role_correlations` tests whether
the roles behave as distinct dimensions, `05_multirole_elite_count`
answers the central question through multiple lenses, `06_role_specialists`
prepares the communication layer, and this notebook brings it together.
What's left is the Tableau dashboard itself, built from these two files,
and the final README/documentation pass.

---

## Conclusión

Se integraron las Fases 2-6 en dos archivos: `dataset_final.csv` (2488
jugadores, 72 columnas que abarcan totales de carrera, tasas per 90, los
cuatro puntajes de rol, conteos de élite en tres umbrales, y el promedio
de puntaje de rol) y `role_specialists.csv` (12 filas de referencia
etiquetadas para el radar chart). Verificado antes de exportar, no
asumido: `role_metrics` y `role_scores` comparten exactamente el mismo
conjunto de 2488 ids de jugador, la unión no produjo duplicados ni
columnas completamente vacías, y cinco jugadores clave (Messi, Maradona,
Cristiano, Cruyff, Zidane) llevan valores consistentes de ambos archivos
fuente, coincidiendo con cada número ya reportado en las Fases 3-6.

El pipeline completo ahora es reproducible de punta a punta desde los
datos crudos de SofaScore hasta estos dos archivos: `01_extraction_coverage`
confirma los datos crudos, `02_population_build` los limpia y agrega,
`03_role_metrics` los normaliza y asigna a roles, `04_role_correlations`
prueba si los roles se comportan como dimensiones distintas,
`05_multirole_elite_count` responde la pregunta central con varias
miradas, `06_role_specialists` prepara la capa de comunicación, y este
notebook lo junta todo. Lo que queda es el dashboard de Tableau en sí,
construido desde estos dos archivos, y la pasada final de README/
documentación.